# Análise Exploratória — Silver: ecommerce_enderecos

Exploração da tabela Silver de endereços (`squad1/silver/ecommerce_enderecos` no ADLS): volume e completude das colunas, distribuições (UF, apelido, endereço principal, CEP, endereços por cliente), taxa de falha das 10 regras de Data Quality e dispersão das coordenadas geográficas por micro-lote.

### 1. Configuração e leitura da Silver
Credenciais do ADLS carregadas do `.env` via API do workspace (no serverless, `load_dotenv()` não acessa arquivos ocultos) e leitura da Delta `squad1/silver/ecommerce_enderecos`.

In [0]:
# 1. Credenciais ADLS (.env via API do workspace) e leitura da Silver
import os
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

ENV_PATH = "/Users/luizpopoff@gmail.com/estagio-empregadados-turma-2/.env"

# No serverless o FUSE não expõe arquivos ocultos: baixa o .env pela API do workspace
_env_txt = WorkspaceClient().workspace.download(ENV_PATH).read().decode()
for _linha in _env_txt.splitlines():
    _linha = _linha.strip()
    if _linha and not _linha.startswith("#") and "=" in _linha:
        _k, _v = _linha.split("=", 1)
        os.environ.setdefault(_k.strip(), _v.strip())

client_id            = os.environ["ADLS_CLIENT_ID"]
tenant_id            = os.environ["ADLS_TENANT_ID"]
client_secret        = os.environ["ADLS_CLIENT_SECRET"]
storage_account_name = os.environ.get("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

silver_path = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net/silver/ecommerce_enderecos"

df_silver = spark.read.format("delta").options(**adls_opts).load(silver_path)

TOTAL = df_silver.count()
print(f"Linhas na Silver: {TOTAL}")
df_silver.printSchema()

### 2. Visão geral e completude
Volume, chaves, micro-lotes ingeridos e quantidade de nulos por coluna de negócio (ignora colunas técnicas `falha_*`, `bronze_*`, `silver_*`).

In [0]:
# 2. Visão geral: volume, chaves, lotes e completude
COLS_NEGOCIO = [c for c in df_silver.columns if not c.startswith(("falha_", "bronze_", "silver_"))]

print(f"Total de linhas:        {TOTAL}")
print(f"Endereços distintos:    {df_silver.select('id_endereco').distinct().count()}")
print(f"Clientes distintos:     {df_silver.select('id_cliente').distinct().count()}")
print(f"Micro-lotes (arquivos): {df_silver.select('bronze_source_file').distinct().count()}")

# Nulos por coluna de negócio (uma única varredura)
_nao_nulos = df_silver.agg(*[F.count(c).alias(c) for c in COLS_NEGOCIO]).collect()[0]
resumo_nulos = spark.createDataFrame(
    [(c, int(TOTAL - _nao_nulos[c])) for c in COLS_NEGOCIO],
    "coluna string, qtd_nulos long",
)
display(resumo_nulos.orderBy(F.desc("qtd_nulos")))

### 3. Distribuições categóricas
UF, apelido, endereço principal, tamanho do CEP (dígitos) e quantidade de endereços/principais por cliente.

In [0]:
# 3. Distribuições: estado, apelido, is_principal, CEP e endereços por cliente
display(df_silver.groupBy("estado").count().orderBy(F.desc("count")))

display(df_silver.groupBy("apelido").count().orderBy(F.desc("count")))

display(df_silver.groupBy("is_principal").count())

# Tamanho do CEP considerando apenas dígitos (regra R03: exatamente 8)
display(
    df_silver
    .select(F.length(F.regexp_replace(F.col("cep").cast("string"), "[^0-9]", "")).alias("tamanho_cep_digitos"))
    .groupBy("tamanho_cep_digitos").count().orderBy("tamanho_cep_digitos")
)

end_por_cliente = df_silver.groupBy("id_cliente").agg(
    F.count("*").alias("qtd_enderecos"),
    F.sum(F.col("is_principal").cast("int")).alias("qtd_principais"),
)
display(end_por_cliente.groupBy("qtd_enderecos").count().orderBy("qtd_enderecos"))
display(end_por_cliente.groupBy("qtd_principais").count().orderBy("qtd_principais"))

### 4. Data Quality
Percentual de linhas com falha (`True`) em cada uma das 10 regras `falha_rNN_*` gravadas pela Silver, ordenado da pior para a melhor.

In [0]:
# 4. Taxa de falha por regra de Data Quality (colunas falha_rNN_*: True = falhou)
REGRAS = [c for c in df_silver.columns if c.startswith("falha_r")]

if not REGRAS:
    print("Nenhuma coluna falha_rNN_* encontrada na Silver.")
else:
    _falhas = df_silver.agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in REGRAS]).collect()[0]

    df_taxas = spark.createDataFrame(
        [(r, int(_falhas[r]), round(_falhas[r] / TOTAL * 100, 2)) for r in REGRAS],
        "regra string, qtd_falhas long, pct_falhas double",
    )
    display(df_taxas.orderBy(F.desc("pct_falhas")))

### 5. Coordenadas e evolução por lote
Estatísticas e dispersão de latitude/longitude, além de registros e linhas com falha por micro-lote (ordem cronológica).

In [0]:
# 5. Coordenadas (estatísticas + dispersão) e registros por micro-lote
import matplotlib.pyplot as plt

display(df_silver.select("latitude", "longitude").summary("count", "min", "25%", "50%", "75%", "max"))

coords = df_silver.select("latitude", "longitude").toPandas()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords["longitude"], coords["latitude"], s=6, alpha=0.4, color="#1f77b4")
ax.set(title="Dispersão dos endereços (lon x lat)", xlabel="longitude", ylabel="latitude")
ax.grid(alpha=0.3)
plt.show()

# Registros e linhas com alguma falha por micro-lote
REGRAS = [c for c in df_silver.columns if c.startswith("falha_r")]
_alguma_falha = F.greatest(*[F.col(c).cast("int") for c in REGRAS]) if REGRAS else F.lit(0)
_lote = F.regexp_extract("bronze_source_file", r"real-time-data/(\d{4}/\d{2}/\d{2}/\d{6})", 1)
display(
    df_silver
    .groupBy(_lote.alias("lote"))
    .agg(
        F.count("*").alias("registros"),
        F.max("silver_processed_at").alias("processado_em"),
        F.sum(_alguma_falha).alias("linhas_com_alguma_falha"),
    )
    .orderBy("lote")
)

In [0]:
%pip install plotly

In [0]:
# 5b. Dispersão dos endereços sobre um mapa do Brasil (plotly interativo, tiles do OpenStreetMap)
import plotly.express as px

# Mantém apenas coordenadas dentro do retângulo delimitador do Brasil
# (exclui nulos e coordenadas inválidas marcadas pela regra R05)
mapa = (
    df_silver
    .filter(
        F.col("latitude").between(-33.75, 5.5)
        & F.col("longitude").between(-73.99, -34.79)
    )
    .select("latitude", "longitude", "estado", "id_endereco")
    .toPandas()
)
print(f"Endereços plotados (coordenadas válidas no Brasil): {len(mapa)} de {TOTAL}")

fig = px.scatter_map(
    mapa,
    lat="latitude",
    lon="longitude",
    hover_name="id_endereco",
    hover_data={"estado": True},
    color="estado",
    zoom=3,
    center={"lat": -14.2, "lon": -51.9},
    height=600,
    title="Dispersão dos endereços — mapa do Brasil",
)
fig.update_layout(map_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()